In [5]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [6]:
def load_data(file_path):
    """Load the Telco Customer Churn dataset."""
    df = pd.read_csv(file_path)

    df = df.drop(columns=["customerID"])

    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

    df = df.dropna(subset=["TotalCharges"])

    df["Churn"] = df["Churn"].map({
        "Yes": 1,
        "No": 0
    })

    return df

In [7]:
def create_features(df):
    service_columns = [
        "PhoneService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    df["num_services"] = 0

    for col in service_columns:
        df["num_services"] += (
            df[col] == "Yes"
        ).astype(int)

    return df

In [8]:
def create_preprocessor(df):
    numeric_features = df.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    numeric_features.remove("Churn")

    categorical_features = df.select_dtypes(
        include=["object"]
    ).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ]
    )

    return preprocessor

In [10]:
# Load and prepare the dataset using our functions

df = load_data(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = create_features(df)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (7032, 21)
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  ...  \
0  No phone service             DSL             No          Yes  ...   
1                No             DSL            Yes           No  ...   
2                No             DSL            Yes          Yes  ...   
3  No phone service             DSL            Yes           No  ...   
4                No     Fiber optic             No           No  ...   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-mo

In [11]:
preprocessor = create_preprocessor(df)

print("Preprocessor created successfully!")
print(preprocessor)

Preprocessor created successfully!
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges', 'num_services']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])


In [12]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5625, 20)
X_test shape: (1407, 20)
y_train shape: (5625,)
y_test shape: (1407,)


In [13]:
# Fit the preprocessor on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (5625, 46)
Processed X_test shape: (1407, 46)


In [14]:
# Engineered feature demonstration

print("Engineered Feature: num_services")

print(
    df[["num_services", "Churn"]].head(10)
)

Engineered Feature: num_services
   num_services  Churn
0             1      0
1             3      0
2             3      1
3             3      0
4             1      1
5             4      1
6             3      0
7             1      0
8             5      1
9             3      0


### Engineered Feature

A new feature named `num_services` was created to represent the number of subscribed services used by each customer. This feature is used as an additional numerical input for the machine learning model.

In [15]:
print("===== FINAL PREPROCESSING SUMMARY =====")

print("Original cleaned dataset:", df.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Processed X_train:", X_train_processed.shape)
print("Processed X_test:", X_test_processed.shape)

print("\nEngineered feature:")
print("num_services")

print("\nTarget encoding:")
print("No  -> 0")
print("Yes -> 1")

===== FINAL PREPROCESSING SUMMARY =====
Original cleaned dataset: (7032, 21)
X_train: (5625, 20)
X_test: (1407, 20)
Processed X_train: (5625, 46)
Processed X_test: (1407, 46)

Engineered feature:
num_services

Target encoding:
No  -> 0
Yes -> 1
